# Runbook 4.4.c: Identifying compromised devices (indicators of compromise)

**Skill:** Detect Attacks · **Maps to:** 4.4.B, 4.4.C · **Language:** Python 3 (pytest)

**Steps:** 1 Brief · 2 Plan · 3 Build · 4 Test · 5 Try it on the evidence · 6 Wrap up

## 1. Brief

**The situation.** Each device under investigation reports a set of observed indicators of compromise (IOCs). Each
IOC type carries a fixed severity weight. Build the scoring engine that sums the weights of the
IOCs observed on a device and returns an overall compromise verdict.

**Lesson objective.** Define an indicator of compromise (IOC), classify common IOC categories, and combine multiple IOCs on a single device into an overall compromise verdict using weighted scoring.

### What comes in
A list of strings, each an IOC type observed on one device. Duplicate entries in the list count
only once (an IOC type either is or is not present on the device).

### What must come out
One of the strings: `"clean"` (total score < 4), `"suspicious"` (4 <= total score < 8),
`"compromised"` (total score >= 8).

- [ ] Read the situation and say in one sentence what the tool must decide
- [ ] Find the input and the output above

## 2. Plan

Work it out on paper before you write code.

### Worked examples
```
["av_disabled", "new_admin_account"]
    -> "compromised"   # 5 + 4 = 9 >= 8

["failed_login_cluster"]
    -> "clean"          # 1 < 4

["new_admin_account", "failed_login_cluster"]
    -> "suspicious"      # 4 + 1 = 5, in [4, 8)
```

- [ ] Work the first example by hand and get the same answer

### Rules to respect
- [ ] Deterministic, offline, standard library only.
- [ ] Duplicate IOC types in the input list must not be double-counted.
- [ ] Unknown/unrecognized IOC type strings contribute 0 and must not raise an error.
- [ ] Thresholds are: `< 4` clean, `4 <= score < 8` suspicious, `>= 8` compromised.
- [ ] Must pass all cases in the tests (visible + hidden).
- [ ] Write the steps your function will take, in plain words
- [ ] List two edge cases (an empty input, a tricky value)

<details><summary>Reference: the exact contract</summary>

Any implementation that satisfies this interface can be swapped in (reference, student, or
composite) and must pass the tests unchanged.

#### Entry point
```python
def solve(iocs: list) -> str
```

#### Input (`iocs`)
A list of `str` IOC type identifiers observed on a single device. Duplicates may appear in the
list but each IOC type is counted at most once toward the total score.

#### IOC weight table
```python
WEIGHTS = {
    "known_malicious_hash": 6,
    "av_disabled": 5,
    "new_admin_account": 4,
    "unusual_outbound_traffic": 3,
    "unexpected_scheduled_task": 3,
    "registry_run_key_added": 2,
    "failed_login_cluster": 1,
}
```
Any string not present in this table contributes a weight of `0`.

#### Output (`result`)
One of `"clean"`, `"suspicious"`, `"compromised"`, based on the total score (sum of unique
matched weights):
- `score < 4` -> `"clean"`
- `4 <= score < 8` -> `"suspicious"`
- `score >= 8` -> `"compromised"`

#### Guarantees / rules
- Pure function, deterministic, no I/O, no network.
- Standard library only unless the hidden grading tests: allow_deps` lists otherwise.
- Duplicate IOC type strings in the input are de-duplicated before scoring.
- Unrecognized IOC strings do not raise; they simply add 0 to the score.
- Thresholds are as specified above (lower bound inclusive, upper bound exclusive per band).

#### Acceptance
Your code passes when every test below shows a checkmark (some are hidden until you run them).

</details>

## 3. Build it

Fill in `solve` below. Run the cell (Shift+Enter) after each change.

- [ ] `solve` handles the worked examples
- [ ] `solve` handles your edge cases

In [ ]:
"""Starter code for 4.4.c. Student completes the TODO.
This file intentionally FAILS the tests until implemented."""


def solve(iocs: list) -> str:
    """Sum the weights of unique observed IOCs and classify a compromise verdict.

    Args:
        iocs: list of str IOC type identifiers observed on one device.

    Returns:
        "clean" (score < 4), "suspicious" (4 <= score < 8), or "compromised" (score >= 8).
    """
    # TODO: implement per the problem statement above
    raise NotImplementedError("implement solve() — see the problem statement above")


## 4. Test it

The tests are the acceptance check, like a change review before a fix goes live.

- [ ] Run the tests
- [ ] Every test says PASS (if not, read the message, go back to step 3)

In [ ]:
import __main__ as impl

def test_example_1_compromised():
    assert impl.solve(["av_disabled", "new_admin_account"]) == "compromised"


def test_example_2_clean():
    assert impl.solve(["failed_login_cluster"]) == "clean"


def test_example_3_suspicious():
    assert impl.solve(["new_admin_account", "failed_login_cluster"]) == "suspicious"


def test_edge_empty_list_is_clean():
    assert impl.solve([]) == "clean"


def test_unknown_ioc_contributes_zero():
    assert impl.solve(["unrecognized_ioc_type"]) == "clean"


def test_duplicate_iocs_not_double_counted():
    assert impl.solve(["av_disabled", "av_disabled", "av_disabled"]) == "suspicious"


def test_boundary_score_exactly_four_is_suspicious():
    assert impl.solve(["new_admin_account"]) == "suspicious"


def test_boundary_score_just_below_four_is_clean():
    assert impl.solve(["unusual_outbound_traffic"]) == "clean"


def test_boundary_score_exactly_eight_is_compromised():
    assert impl.solve(["av_disabled", "unusual_outbound_traffic"]) == "compromised"


def test_boundary_score_seven_is_suspicious():
    assert impl.solve(["unusual_outbound_traffic", "unexpected_scheduled_task",
                        "failed_login_cluster"]) == "suspicious"


def test_high_severity_combination_is_compromised():
    assert impl.solve(["known_malicious_hash", "unusual_outbound_traffic",
                        "registry_run_key_added"]) == "compromised"


def test_all_iocs_combined_is_compromised():
    all_iocs = ["known_malicious_hash", "av_disabled", "new_admin_account",
                "unusual_outbound_traffic", "unexpected_scheduled_task",
                "registry_run_key_added", "failed_login_cluster"]
    assert impl.solve(all_iocs) == "compromised"


# ---- run every test above ------------------------------------------------
_results = []
for _name, _fn in list(globals().items()):
    if _name.startswith("test_") and callable(_fn):
        try:
            _fn()
            _results.append((_name, None))
        except Exception as _e:
            _results.append((_name, _e))
for _name, _e in _results:
    print(("PASS  " if _e is None else "FAIL  ") + _name + ("" if _e is None else f": {_e!r}"))
print(f"{sum(e is None for _, e in _results)}/{len(_results)} tests passed")


## 5. Try it on the evidence

Run your finished tool on the examples from the brief, then change one value and predict the answer before you run it again.

- [ ] Every example says OK
- [ ] Your own changed example gives the answer you predicted

In [ ]:
checks = [
    (lambda: solve(["av_disabled", "new_admin_account"]),
     "compromised"),
    (lambda: solve(["failed_login_cluster"]),
     "clean"),
    (lambda: solve(["new_admin_account", "failed_login_cluster"]),
     "suspicious"),
]

for n, (run, want) in enumerate(checks, 1):
    try:
        got = run()
    except Exception as e:
        got = "error: " + repr(e)
    print("example", n, "OK  " if got == want else "DIFF", "got:", got, " expected:", want)


## 6. Wrap up

- [ ] Explain in one sentence why your rule catches the problem
- [ ] Name one way an attacker could slip past it (a false negative)
- [ ] Name one harmless input it might wrongly flag (a false positive)
- [ ] Say where this check would run in a real organisation, and who would act on it